Optical flow - RAFT


In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/RAFT"):
    subprocess.run(["git", "clone", "https://github.com/princeton-vl/RAFT.git", "/kaggle/working/RAFT"], check=True)

os.chdir("/kaggle/working/RAFT")
subprocess.run(["pip", "-q", "install", "opencv-python"], check=True)

if not os.path.exists("/kaggle/working/RAFT/models/raft-small.pth"):
    subprocess.run(["bash", "download_models.sh"], check=True)

subprocess.run(["find", "/kaggle/working/RAFT/models", "-name", "*.pth"], check=True)

In [ ]:
import sys
import os
import glob
import cv2
import torch
from argparse import Namespace

sys.path.append("/kaggle/working/RAFT")
sys.path.append("/kaggle/working/RAFT/core")

from raft import RAFT
from utils.utils import InputPadder
from utils.flow_viz import flow_to_image

INPUT_ROOT = "/kaggle/input/datasets/dharun235/lateral-insect-view/Lateral view insect video"
OUTPUT_ROOT = "/kaggle/working/RAFT_Outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

VIDEO_EXTENSIONS = ("*.mp4", "*.avi", "*.mov", "*.mkv")
MAX_DIM = 640
ITERS = 12
SKIP_EXISTING = True

device = "cuda" if torch.cuda.is_available() else "cpu"

args = Namespace(
    small=True,
    mixed_precision=True,
    alternate_corr=False,
    dropout=0,
    corr_levels=4,
    corr_radius=3,
)

model = torch.nn.DataParallel(RAFT(args))
ckpt = "/kaggle/working/RAFT/models/raft-small.pth"
model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False))
model = model.module.to(device)
model.eval()

print("RAFT-small loaded.")
print("Device:", device)


def resize_for_raft(frame):
    h, w = frame.shape[:2]
    scale = min(1.0, MAX_DIM / max(h, w))
    if scale == 1.0:
        return frame
    new_w = int(w * scale) // 8 * 8
    new_h = int(h * scale) // 8 * 8
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)


def preprocess(frame):
    return torch.from_numpy(frame).permute(2, 0, 1).float()[None].to(device)


video_list = sorted(
    video_path
    for pattern in VIDEO_EXTENSIONS
    for video_path in glob.glob(os.path.join(INPUT_ROOT, "**", pattern), recursive=True)
)

print(f"Found {len(video_list)} videos.")

for idx, video_path in enumerate(video_list, 1):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_video = os.path.join(OUTPUT_ROOT, f"{video_name}_raft.mp4")

    if SKIP_EXISTING and os.path.exists(output_video):
        print(f"[{idx}/{len(video_list)}] Skipping existing: {output_video}")
        continue

    print("=" * 80)
    print(f"[{idx}/{len(video_list)}] Processing: {video_path}")

    cap = cv2.VideoCapture(video_path)
    ret, frame1 = cap.read()
    if not ret:
        print("Could not read video. Skipping.")
        cap.release()
        continue

    frame1 = resize_for_raft(frame1)
    h, w = frame1.shape[:2]
    fps = cap.get(cv2.CAP_PROP_FPS) or 30

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create MP4 writer: {output_video}")

    frame_count = 0

    while True:
        ret, frame2 = cap.read()
        if not ret:
            break

        frame2 = resize_for_raft(frame2)
        image1 = preprocess(frame1)
        image2 = preprocess(frame2)

        padder = InputPadder(image1.shape)
        image1, image2 = padder.pad(image1, image2)

        with torch.no_grad():
            _, flow = model(image1, image2, iters=ITERS, test_mode=True)

        flow = padder.unpad(flow[0]).permute(1, 2, 0).cpu().numpy()
        writer.write(flow_to_image(flow))

        frame1 = frame2
        frame_count += 1

        del image1, image2, flow
        if device == "cuda":
            torch.cuda.empty_cache()

        if frame_count % 500 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    writer.release()
    print("Saved:", output_video)

print("Done. Outputs:", OUTPUT_ROOT)